# Applied NLP — Lab 4: Full NLP Pipeline + Mini Interactive Application

**Dataset:** NLTK Movie Reviews Corpus (2000 samples: 1000 positive / 1000 negative movie reviews)

This notebook covers, in order: dataset understanding, preprocessing, linguistic analysis (POS + dependency parsing),
text statistics & N-grams, a Bigram Language Model (MLE + Laplace smoothing), perplexity evaluation, and a Gradio
mini application that uses the pipeline and the Bigram model built from this dataset (no pretrained autocomplete model is used).

In [ ]:
# Install required libraries (Colab)
!pip install -q spacy gradio nltk
!python -m spacy download en_core_web_sm -q

In [ ]:
import pandas as pd
import numpy as np
import re
import math
import random
from collections import Counter, defaultdict

import nltk
from nltk.corpus import movie_reviews, stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.util import ngrams

import spacy
from spacy import displacy

import gradio as gr

# NLTK downloads
for pkg in ["movie_reviews", "punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(pkg, "->", e)

nlp = spacy.load("en_core_web_sm")

random.seed(42)
np.random.seed(42)

## Part 1 — Dataset Understanding

In [ ]:
# Build a DataFrame from the NLTK Movie Reviews corpus
rows = []
for category in movie_reviews.categories():
    for fileid in movie_reviews.fileids(category):
        rows.append({"text": movie_reviews.raw(fileid), "label": category})

df = pd.DataFrame(rows)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

In [ ]:
# Missing values and duplicate records
print("Missing values per column:\n", df.isnull().sum())
print("\nNumber of duplicate rows:", df.duplicated().sum())

# Main text column + text length
df["text_length"] = df["text"].apply(lambda x: len(x.split()))
df[["text", "text_length"]].head()

**Dataset description:** The dataset contains 2,000 English movie reviews, each labeled as `pos` (positive) or
`neg` (negative). Reviews vary in length (tens to hundreds of words).

**Possible NLP task:** Sentiment classification (predicting `pos`/`neg` from the review text). We will additionally
use this text data to build a general-purpose N-gram language model for the second half of the lab.

## Part 2 — Text Preprocessing

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)          # remove URLs
    text = re.sub(r"\S+@\S+", " ", text)                  # remove emails
    text = re.sub(r"[^a-z\s]", " ", text)                  # remove punctuation/digits/symbols
    text = re.sub(r"\s+", " ", text).strip()               # collapse whitespace
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words]
    return " ".join(tokens)

df["original_text"] = df["text"]
df["processed_text"] = df["text"].apply(preprocess_text)

df[["original_text", "processed_text"]].head()

In [ ]:
# Show 5 examples before/after preprocessing
for i in range(5):
    print(f"--- Example {i+1} ---")
    print("ORIGINAL :", df["original_text"].iloc[i][:200].replace("\n", " "))
    print("PROCESSED:", df["processed_text"].iloc[i][:200])
    print()

**Preprocessing decision that affects meaning:** Our stopword list removes the word *"not"* along with other
common words. For sentiment-bearing text this can flip the apparent meaning — e.g. *"not good"* becomes just
*"good"* after stopword removal, which loses the negation. In a task where polarity matters (like sentiment
analysis), it would be safer to keep negation words (`not`, `no`, `never`) out of the stopword list.

## Part 3 — Linguistic Analysis

In [ ]:
# Pick 5 sample reviews and run POS tagging + lemmatization with spaCy
sample_texts = df["original_text"].sample(5, random_state=42).tolist()

for i, text in enumerate(sample_texts):
    doc = nlp(text[:300])  # limit to first ~300 chars per sample for readability
    print(f"=== Sample {i+1} ===")
    for token in doc:
        if token.is_alpha:
            print(f"{token.text:<15}{token.pos_:<8}lemma: {token.lemma_}")
    print()

In [ ]:
# Dependency parsing visualization for one sentence using spaCy displaCy
doc_sample = nlp(sample_texts[0])
first_sentence = list(doc_sample.sents)[0]
displacy.render(first_sentence, style="dep", jupyter=True, options={"distance": 100})

**What dependency parsing adds beyond tokenization:** Tokenization only splits text into individual words.
Dependency parsing additionally reveals the *grammatical relationships between* those words — e.g. which word is
the subject of a verb, which words modify a noun, which phrase is the object of an action. This structural
information is essential for tasks like information extraction, question answering, and understanding *who did
what to whom* in a sentence.

## Part 4 — Text Statistics and N-Grams

In [ ]:
# 10 most frequent words across the processed dataset
all_tokens = " ".join(df["processed_text"]).split()
word_freq = Counter(all_tokens)
print("Top 10 most frequent words:")
for word, count in word_freq.most_common(10):
    print(f"  {word:<15}{count}")

In [ ]:
# Unigrams, Bigrams, Trigrams over the processed dataset
unigrams  = list(ngrams(all_tokens, 1))
bigrams_  = list(ngrams(all_tokens, 2))
trigrams_ = list(ngrams(all_tokens, 3))

uni_freq = Counter(unigrams)
bi_freq  = Counter(bigrams_)
tri_freq = Counter(trigrams_)

print("Top 10 Unigrams:")
for gram, c in uni_freq.most_common(10):
    print(" ", gram, c)

print("\nTop 10 Bigrams:")
for gram, c in bi_freq.most_common(10):
    print(" ", gram, c)

print("\nTop 10 Trigrams:")
for gram, c in tri_freq.most_common(10):
    print(" ", gram, c)

**Effect of increasing n:** A unigram only captures individual word frequency, with no context about neighboring
words. A bigram captures the relationship between two consecutive words (some local context, e.g. common phrase
starts). A trigram captures even more context (three-word patterns), which produces more meaningful and specific
phrases but also increases sparsity — many trigrams will only occur once or never, since the number of possible
combinations grows quickly as n increases.

## Part 5 — Bigram Language Model

In [ ]:
# Build the bigram model from the processed dataset (tokenized per document so we don't
# create fake bigrams across unrelated reviews)
tokenized_docs = [doc.split() for doc in df["processed_text"]]

unigram_counts = Counter()
bigram_counts = Counter()
next_word_counts = defaultdict(Counter)

for tokens in tokenized_docs:
    unigram_counts.update(tokens)
    for w1, w2 in zip(tokens, tokens[1:]):
        bigram_counts[(w1, w2)] += 1
        next_word_counts[w1][w2] += 1

vocab = set(unigram_counts.keys())
V = len(vocab)
print("Vocabulary size:", V)
print("Number of distinct bigrams:", len(bigram_counts))

In [ ]:
def bigram_mle_prob(w1, w2):
    """Maximum Likelihood Estimate: P(w2 | w1) = count(w1,w2) / count(w1)"""
    if unigram_counts[w1] == 0:
        return 0.0
    return bigram_counts[(w1, w2)] / unigram_counts[w1]

def bigram_laplace_prob(w1, w2):
    """Add-one (Laplace) smoothed probability"""
    return (bigram_counts[(w1, w2)] + 1) / (unigram_counts[w1] + V)

# At least 3 example bigram probabilities (using the most frequent bigrams)
print("Example MLE bigram probabilities:")
for (w1, w2), _ in bi_freq.most_common(3):
    print(f"  P({w2!r} | {w1!r}) = {bigram_mle_prob(w1, w2):.4f}")

In [ ]:
# Find one unseen bigram (a pair that never occurs together) and show MLE = 0
most_common_word = unigram_counts.most_common(1)[0][0]
unseen_pair = None
for w in vocab:
    if bigram_counts[(most_common_word, w)] == 0:
        unseen_pair = (most_common_word, w)
        break

print("Unseen bigram:", unseen_pair)
print("MLE probability          :", bigram_mle_prob(*unseen_pair))
print("Laplace-smoothed probability:", bigram_laplace_prob(*unseen_pair))

In [ ]:
def predict_next_word(word, top_n=3):
    """Return the top-N most likely next words after `word`, using bigram MLE probabilities."""
    word = word.lower()
    if word not in next_word_counts:
        return []
    total = unigram_counts[word]
    top = next_word_counts[word].most_common(top_n)
    return [(w2, count / total) for w2, count in top]

# Demo
print(predict_next_word(most_common_word))
print(predict_next_word("movie"))

## Part 6 — Perplexity

In [ ]:
def sentence_perplexity(sentence, smoothing=True):
    tokens = preprocess_text(sentence).split()
    N = len(tokens)
    if N < 2:
        return float("inf")

    log_prob_sum = 0.0
    for w1, w2 in zip(tokens, tokens[1:]):
        if smoothing:
            p = (bigram_counts[(w1, w2)] + 1) / (unigram_counts.get(w1, 0) + V)
        else:
            p = bigram_mle_prob(w1, w2)
            if p == 0:
                p = 1e-10  # avoid log(0) for the unsmoothed case
        log_prob_sum += math.log(p)

    return math.exp(-log_prob_sum / (N - 1))

# Two test sentences: one similar in style to movie reviews, one with uncommon word combinations
sentence_common   = "the movie was really good and the acting was great"
sentence_uncommon = "the quantum spaceship whispered lavender equations to the moon"

pp_common   = sentence_perplexity(sentence_common)
pp_uncommon = sentence_perplexity(sentence_uncommon)

print(f"Perplexity (dataset-like sentence) : {pp_common:.2f}")
print(f"Perplexity (uncommon sentence)     : {pp_uncommon:.2f}")

**Which is predicted better?** The dataset-like sentence should get the **lower** perplexity, because its word
pairs occur more often (or are more similar to patterns) in the movie-review training data, so the model assigns
them higher probability. The uncommon sentence uses word combinations the model rarely or never saw, so its
probability is lower and perplexity is higher — this is exactly what perplexity measures: how "surprised" the
model is by a sequence, given what it learned during training.

## Part 7 — Mini NLP Application (Gradio)

In [ ]:
def nlp_mini_app(user_text):
    if not user_text or not user_text.strip():
        return "", "", "", "Please enter some text."

    processed = preprocess_text(user_text)

    doc = nlp(user_text)
    pos_lines = [f"{t.text} -> POS: {t.pos_}, Lemma: {t.lemma_}" for t in doc if t.is_alpha]
    pos_output = "\n".join(pos_lines) if pos_lines else "No tokens found."

    proc_tokens = processed.split()
    last_word = proc_tokens[-1] if proc_tokens else ""
    predictions = predict_next_word(last_word)
    if predictions:
        pred_output = "\n".join([f"{w}: {p:.4f}" for w, p in predictions])
    else:
        pred_output = f"No suggestions available for '{last_word}' (word not seen enough in training data)."

    return user_text, processed, pos_output, pred_output


demo = gr.Interface(
    fn=nlp_mini_app,
    inputs=gr.Textbox(label="Enter a sentence", placeholder="e.g. this movie was absolutely"),
    outputs=[
        gr.Textbox(label="Original Text"),
        gr.Textbox(label="Processed Text"),
        gr.Textbox(label="Tokens / POS / Lemma", lines=8),
        gr.Textbox(label="Top Next-Word Suggestions (Bigram Model)"),
    ],
    title="Mini NLP Application — Movie Reviews Bigram Model",
    description="Uses the preprocessing pipeline and the from-scratch Bigram Language Model built earlier in this notebook (no pretrained autocomplete model).",
)

demo.launch(share=True, debug=False)

## Final Reflection

1. **What preprocessing step had the greatest effect on your dataset?**
   Lowercasing combined with punctuation/number removal reduced vocabulary size the most, since it merged many
   surface variants of the same word into one form. Stopword removal also had a large effect on the bigram model
   because it removed many high-frequency filler words, which changed which bigrams became most common.

2. **What additional information did POS tagging or dependency parsing provide?**
   POS tagging showed the grammatical role of each word (noun, verb, adjective, etc.), which raw tokens alone don't
   convey. Dependency parsing went further by showing how words relate to each other structurally (subject, object,
   modifier), which is useful for understanding sentence meaning, not just word identity.

3. **What is the main limitation of an N-gram language model?**
   It only looks at a fixed, short window of previous words (here, just one word for a bigram model), so it cannot
   capture long-range context or true semantic meaning. It also suffers from data sparsity — many valid word
   sequences never appeared in training, so the model assigns them zero (or near-zero, with smoothing) probability
   even if they are perfectly natural sentences.

4. **If you had more time, how would you improve your mini NLP application?**
   I would extend the bigram model to a trigram (or higher-order) model with proper backoff/interpolation, add
   sentence-boundary tokens so predictions respect where sentences start and end, and let the app compare
   suggestions from N-grams of different orders side by side.